In [ ]:
from pathlib import Path

import torch
from torchvision import datasets, transforms

_cwd = Path.cwd()
if (_cwd / "data").is_dir():
    DATA_ROOT = str(_cwd / "data")
elif (_cwd.parent / "data").is_dir():
    DATA_ROOT = str(_cwd.parent / "data")
else:
    DATA_ROOT = "./data"

eval_transform = transforms.Compose(
    [
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
test_dataset = datasets.INaturalist(
    root=DATA_ROOT,
    version="2019",
    transform=eval_transform,
    download=True,
)

NUM_CLASSES = len(test_dataset.all_categories)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

RESULTS_PATH = (
    _cwd / "testing" / "dataset_eval_results.txt"
    if (_cwd / "testing").is_dir()
    else _cwd / "dataset_eval_results.txt"
)

In [ ]:
import torch.nn as nn
import torchvision.models as models
from torch.profiler import ProfilerActivity, profile, record_function
from torchvision.models import EfficientNet_B0_Weights

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)
model.eval()

total, correct = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        pred = logits.argmax(dim=1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()

test_accuracy_pct = 100.0 * correct / max(total, 1)

# Total parameters (all are used each forward for this dense model)
total_params = sum(p.numel() for p in model.parameters())
activated_params = total_params

# FLOPs / MACs for one forward pass (batch_size=1) via PyTorch profiler
acts = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    acts.append(ProfilerActivity.CUDA)

dummy = torch.randn(1, 3, 224, 224, device=device)
flops_one_forward = None
model.eval()
with torch.no_grad():
    with profile(activities=acts, with_flops=True, record_shapes=True) as prof:
        with record_function("forward"):
            model(dummy)

flops_sum = sum(
    int(e.flops) for e in prof.key_averages() if getattr(e, "flops", None) is not None
)
if flops_sum > 0:
    flops_one_forward = flops_sum
else:
    # Many backends (e.g. MPS) omit FLOPs; repeat on CPU for a comparable count
    cpu_m = model.to("cpu")
    cx = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        with profile(activities=[ProfilerActivity.CPU], with_flops=True) as prof_cpu:
            with record_function("forward"):
                cpu_m(cx)
    flops_one_forward = sum(
        int(e.flops)
        for e in prof_cpu.key_averages()
        if getattr(e, "flops", None) is not None
    )
    model.to(device)

lines = [
    f"model: efficientnet_b0 (classifier out_features={NUM_CLASSES})",
    f"test_samples: {total}",
    f"test_accuracy_percent: {test_accuracy_pct:.4f}",
    f"total_parameters: {total_params}",
    f"activated_parameters_per_inference: {activated_params}",
    f"flops_per_inference_batch1: {flops_one_forward}",
    "",
    "Notes:",
    "- For this dense EfficientNet, every parameter participates in each forward pass;",
    "  activated_parameters_per_inference equals total_parameters.",
    "- flops_per_inference_batch1 comes from torch.profiler (with_flops=True).",
    "  If still zero, this PyTorch build may not implement FLOP counting for these ops.",
]

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text("\n".join(lines), encoding="utf-8")

print(f"Accuracy on {total} images: {test_accuracy_pct:.4f}%")
print(f"Total / activated parameters: {total_params:,}")
print(f"FLOPs (batch=1, profiler): {flops_one_forward:,}")
print(f"Wrote: {RESULTS_PATH.resolve()}")